# MOHIM repeated motif → full-song Cover-NoFSQ LoRA

원곡 오디오를 단일 target으로 학습합니다. `Source 64ch = raw 반복 motif`, `Mask 64ch = 1`, `Target 64ch = noised full-song`을 유지하되 text condition에는 ACE-Step Cover instruction을 사용합니다.

## 0. Drive와 저장소 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'codex/full-song-motif-source'
REPO_DIR = Path('/content/MOHIM')
if not (REPO_DIR / '.git').is_dir():
    !git clone -b "{BRANCH}" "{REPOSITORY}" "{REPO_DIR}"
    if _exit_code != 0:
        raise RuntimeError(f'git clone 실패: {_exit_code}')
%cd {REPO_DIR}
!git fetch origin "{BRANCH}"
if _exit_code != 0:
    raise RuntimeError(f'git fetch 실패: {_exit_code}')
!git switch "{BRANCH}"
if _exit_code != 0:
    raise RuntimeError(f'git switch 실패: {_exit_code}')
!git pull --ff-only origin "{BRANCH}"
if _exit_code != 0:
    raise RuntimeError(f'git pull 실패: {_exit_code}')
print('repository:', REPO_DIR)

In [ ]:
%pip uninstall -y torchao

# torchao/flash-attn은 필수가 아니며 Colab torch 조합과 충돌할 수 있어 제외합니다.
def write_filtered_requirements(requirements_path, output_path):
    excluded = ('torchao', 'flash-attn')
    lines = requirements_path.read_text(encoding='utf-8').splitlines()
    kept = [line for line in lines if not any(name in line.lower() for name in excluded)]
    output_path.write_text('\n'.join(kept) + '\n', encoding='utf-8')

root_requirements = Path('/tmp/mohim_full_song_requirements.txt')
write_filtered_requirements(REPO_DIR / 'requirements.txt', root_requirements)
%pip install -q -r {root_requirements}
from mohim.trainer import DEFAULT_REVISION, apply_acestep_patch, ensure_acestep_repo
ACESTEP_DIR = ensure_acestep_repo(Path('/content/ACE-Step-1.5'), revision=DEFAULT_REVISION)
apply_acestep_patch(ACESTEP_DIR, REPO_DIR / 'patches/ace-step-1.5-dual-stream.patch')
acestep_requirements = Path('/tmp/mohim_acestep_full_song_requirements.txt')
write_filtered_requirements(ACESTEP_DIR / 'requirements.txt', acestep_requirements)
%pip install -q -r {acestep_requirements}
# ACE-Step package metadata는 Python 3.13을 거부하므로 editable install 대신 소스를 직접 사용합니다.
ace_source = str(ACESTEP_DIR)
if ace_source not in sys.path:
    sys.path.insert(0, ace_source)
existing_pythonpath = os.environ.get('PYTHONPATH', '')
pythonpath_entries = [entry for entry in existing_pythonpath.split(os.pathsep) if entry]
if ace_source not in pythonpath_entries:
    os.environ['PYTHONPATH'] = os.pathsep.join([ace_source, *pythonpath_entries])
print('ACE-Step:', ACESTEP_DIR)

## 1. 분리된 v8 경로와 full-song manifest

In [ ]:
import json
from mohim.manifest import build_full_song_manifest

LORA_VERSION = 'v8_full_song_cover_nofsq_captioned'
DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset_mean_centered')
MOTIF_VERSION_PATH = DATASET_DIR / '.mohim_motif_version'
MANIFEST_PATH = DATASET_DIR / 'full_song_cover_nofsq_manifest.json'
CAPTION_CACHE_PATH = DATASET_DIR / 'ace_full_song_caption_cache.json'
AUDIO_CODE_CACHE_PATH = DATASET_DIR / 'ace_full_song_audio_code_cache.json'
BASE_TENSOR_DIR = Path('/content/drive/MyDrive/MOHIM/full_song_tensors_repeated_motif')
TENSOR_DIR = Path('/content/drive/MyDrive/MOHIM/full_song_tensors_cover_nofsq_captioned')
TENSOR_SCHEMA_PATH = TENSOR_DIR / '.mohim_schema'
CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/checkpoints')
LORA_RUN_DIR = Path('/content/drive/MyDrive/MOHIM/lora_run') / LORA_VERSION
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/mohim_lora_checkpoints') / LORA_VERSION
INFERENCE_OUTPUT_DIR = Path('/content/drive/MyDrive/MOHIM/inference') / LORA_VERSION
MODEL_VARIANT = 'base'
LM_MODEL_NAME = 'acestep-5Hz-lm-4B'
MAX_DURATION = 300.0
DEVICE = 'cuda'
RESET_TENSORS = False
RESET_LORA_RUN = False
CAPTION_LIMIT = None  # 전체 실행은 None, captioner 확인만 하려면 3

assert MOTIF_VERSION_PATH.is_file(), f'motif dataset version이 없습니다: {MOTIF_VERSION_PATH}'
motif_dataset_config = json.loads(MOTIF_VERSION_PATH.read_text(encoding='utf-8'))
allowed_track_ids = motif_dataset_config.get('accepted_track_ids')
assert isinstance(allowed_track_ids, list) and allowed_track_ids, '학습 가능한 track 목록이 없습니다.'
if MANIFEST_PATH.is_file():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    print('기존 manifest 재사용:', MANIFEST_PATH)
else:
    manifest = build_full_song_manifest(DATASET_DIR, MANIFEST_PATH, allowed_track_ids=allowed_track_ids)
    print('새 manifest 저장:', MANIFEST_PATH)
assert manifest['metadata']['num_samples'] > 0, 'full-song manifest가 비었습니다.'
assert not manifest['metadata']['skipped'], f'원곡 경로가 없어서 제외된 곡이 있습니다: {manifest["metadata"]["skipped"][:10]}'
print('samples:', manifest['metadata']['num_samples'])
print(json.dumps(manifest['samples'][0], ensure_ascii=False, indent=2))

## 1.1 ACE-Step full-song captioner

원곡 전체를 ACE-Step semantic codes로 변환한 뒤 내장 5Hz LM으로 caption만 생성합니다. BPM·key·박자·genre·language 출력은 conditioning에 사용하지 않습니다. 결과는 매 곡마다 Drive에 저장되어 중단 후 이어서 실행할 수 있습니다.

In [ ]:
import contextlib
import gc
import io
import torch
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.model_downloader import ensure_lm_model

# 이전 실행이 OOM으로 끝났다면 남아 있는 handler부터 제거합니다.
for old_handler_name in ('llm_handler', 'dit_handler'):
    old_handler = globals().pop(old_handler_name, None)
    if old_handler is not None and hasattr(old_handler, 'unload'):
        old_handler.unload()
    del old_handler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

CAPTION_CACHE_SCHEMA = 'ace_full_song_caption_only_recurring_motif_v1'
AUDIO_CODE_CACHE_SCHEMA = 'ace_full_song_audio_codes_v1'
if CAPTION_CACHE_PATH.is_file():
    caption_cache = json.loads(CAPTION_CACHE_PATH.read_text(encoding='utf-8'))
    assert caption_cache.get('schema') == CAPTION_CACHE_SCHEMA, f'caption cache schema 불일치: {CAPTION_CACHE_PATH}'
else:
    caption_cache = {'schema': CAPTION_CACHE_SCHEMA, 'captions': {}}
captions_by_track = caption_cache['captions']
if AUDIO_CODE_CACHE_PATH.is_file():
    audio_code_cache = json.loads(AUDIO_CODE_CACHE_PATH.read_text(encoding='utf-8'))
    assert audio_code_cache.get('schema') == AUDIO_CODE_CACHE_SCHEMA, f'audio code cache schema 불일치: {AUDIO_CODE_CACHE_PATH}'
else:
    audio_code_cache = {'schema': AUDIO_CODE_CACHE_SCHEMA, 'audio_codes': {}}
audio_codes_by_track = audio_code_cache['audio_codes']

def save_caption_cache():
    temporary = CAPTION_CACHE_PATH.with_name(CAPTION_CACHE_PATH.name + '.tmp')
    temporary.write_text(json.dumps(caption_cache, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(CAPTION_CACHE_PATH)

def save_audio_code_cache():
    temporary = AUDIO_CODE_CACHE_PATH.with_name(AUDIO_CODE_CACHE_PATH.name + '.tmp')
    temporary.write_text(json.dumps(audio_code_cache, ensure_ascii=False) + '\n', encoding='utf-8')
    temporary.replace(AUDIO_CODE_CACHE_PATH)

def recurring_motif_sentence(motif_stem):
    stem = str(motif_stem).strip().lower().replace('_', ' ')
    return (
        f'This full song is built around the provided {stem} motif. '
        f'The clear, distinctive, and recognizable {stem} motif serves as '
        "the song's central musical hook and recurring signature, returning "
        'prominently and frequently throughout the entire song—not only in the '
        'intro, but across verses, choruses, transitions, and instrumental passages—'
        'while remaining clearly audible within the full arrangement and creating '
        'a strong sense of continuity.'
    )

def understand_caption_only(handler, audio_codes):
    formatted_prompt = handler.build_formatted_prompt_for_understanding(audio_codes)
    output_text, status = handler.generate_from_formatted_prompt(
        formatted_prompt=formatted_prompt,
        cfg={
            'temperature': 0.3,
            'top_k': None,
            'top_p': None,
            'repetition_penalty': 1.0,
            'target_duration': None,
            'user_metadata': None,
            'skip_caption': False,
            'skip_language': True,
            'skip_genres': True,
            'generation_phase': 'understand',
            'caption': '',
            'lyrics': '',
        },
        use_constrained_decoding=True,
        constrained_decoding_debug=False,
        stop_at_reasoning=True,
    )
    if not output_text:
        return {}, status
    metadata, _ = handler.parse_lm_output(output_text)
    return metadata, status

pending_samples = [sample for sample in manifest['samples'] if str(sample['track_id']) not in captions_by_track]
selected_samples = pending_samples if CAPTION_LIMIT is None else pending_samples[:CAPTION_LIMIT]
print(f'caption cache: {len(captions_by_track)}/{len(manifest["samples"])}')
print(f'this run: {len(selected_samples)}')

if selected_samples:
    lm_download_ok, lm_download_status = ensure_lm_model(
        model_name=LM_MODEL_NAME, checkpoints_dir=CHECKPOINT_DIR,
    )
    assert lm_download_ok, lm_download_status
    os.environ['ACESTEP_CHECKPOINTS_DIR'] = str(CHECKPOINT_DIR)

    # Phase 1: 원곡을 semantic audio codes로 변환해 Drive에 캐시합니다.
    code_samples = [sample for sample in selected_samples if str(sample['track_id']) not in audio_codes_by_track]
    print(f'audio code cache: {len(audio_codes_by_track)}/{len(manifest["samples"])}; extract now: {len(code_samples)}')
    code_failures = []
    if code_samples:
        dit_handler = AceStepHandler()
        dit_status, dit_ok = dit_handler.initialize_service(
            project_root=str(CHECKPOINT_DIR.parent), config_path=f'acestep-v15-{MODEL_VARIANT}',
            device=DEVICE, use_flash_attention=False, compile_model=False,
            offload_to_cpu=False, offload_dit_to_cpu=True, use_mlx_dit=False,
        )
        assert dit_ok, dit_status
        for index, sample in enumerate(code_samples, 1):
            track_id = str(sample['track_id'])
            try:
                audio_codes = dit_handler.convert_src_audio_to_codes(str(sample['audio_path']))
                if not audio_codes or str(audio_codes).startswith('❌'):
                    raise RuntimeError(f'semantic code 변환 실패: {audio_codes}')
                audio_codes_by_track[track_id] = str(audio_codes)
                save_audio_code_cache()
                print(f'[AUDIO CODES {index}/{len(code_samples)}] {track_id}')
            except Exception as exc:
                code_failures.append(f'{track_id}: {exc}')
                print(f'[AUDIO CODES FAILED {index}/{len(code_samples)}] {code_failures[-1]}')

        del dit_handler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            free_bytes, total_bytes = torch.cuda.mem_get_info()
            print(f'ACE-Step unloaded; GPU free: {free_bytes / 2**30:.2f}/{total_bytes / 2**30:.2f} GiB')
    if code_failures:
        raise RuntimeError(f'audio code 변환 실패 {len(code_failures)}곡: {code_failures[:10]}')

    # Phase 2: ACE-Step 본체가 내려간 GPU에 4B LM만 올려 caption을 생성합니다.
    llm_handler = LLMHandler()
    llm_status, llm_ok = llm_handler.initialize(
        checkpoint_dir=str(CHECKPOINT_DIR), lm_model_path=LM_MODEL_NAME,
        backend='pt', device=DEVICE, offload_to_cpu=False, dtype=torch.bfloat16,
    )
    assert llm_ok, llm_status

    caption_failures = []
    for index, sample in enumerate(selected_samples, 1):
        track_id = str(sample['track_id'])
        audio_path = str(sample['audio_path'])
        try:
            audio_codes = audio_codes_by_track[track_id]
            with contextlib.redirect_stdout(io.StringIO()):
                caption_metadata, caption_status = understand_caption_only(llm_handler, audio_codes)
            ace_caption = str(caption_metadata.get('caption', '')).strip()
            if not ace_caption:
                raise RuntimeError(f'caption 생성 실패: {caption_status}')
            final_caption = f'{ace_caption} {recurring_motif_sentence(sample["motif_stem"])}'
            captions_by_track[track_id] = {
                'ace_caption': ace_caption, 'caption': final_caption,
                'audio_path': audio_path, 'motif_stem': sample['motif_stem'],
            }
            save_caption_cache()
            print(f'[CAPTION {index}/{len(selected_samples)}] {track_id}: {final_caption}')
        except Exception as exc:
            caption_failures.append(f'{track_id}: {exc}')
            print(f'[CAPTION FAILED {index}/{len(selected_samples)}] {caption_failures[-1]}')

    llm_handler.unload()
    del llm_handler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if caption_failures:
        raise RuntimeError(f'caption 실패 {len(caption_failures)}곡: {caption_failures[:10]}')

missing_captions = [str(sample['track_id']) for sample in manifest['samples'] if str(sample['track_id']) not in captions_by_track]
if missing_captions:
    raise RuntimeError(
        f'caption이 {len(missing_captions)}곡 남았습니다. preview였다면 CAPTION_LIMIT=None으로 바꾸고 이 셀을 다시 실행하세요.'
    )
for sample in manifest['samples']:
    sample['caption'] = captions_by_track[str(sample['track_id'])]['caption']
    sample['bpm'] = None
    sample['keyscale'] = ''
    sample['timesignature'] = ''
manifest['metadata']['caption_schema'] = CAPTION_CACHE_SCHEMA
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('[OK] full-song captions:', len(captions_by_track))
print(json.dumps(manifest['samples'][0], ensure_ascii=False, indent=2))

## 2. 기존 audio latent 재사용 + Cover instruction condition 재인코딩

v7의 full-song target과 raw 반복 motif context는 그대로 복사합니다. Caption과 Cover instruction에 영향받는 text/lyric/DiT encoder condition만 두 단계로 다시 계산합니다.

In [ ]:
import hashlib
import shutil
import torch
from acestep.constants import SFT_GEN_PROMPT, TASK_INSTRUCTIONS
from acestep.training.dataset_builder_modules.preprocess_encoder import run_encoder
from acestep.training.dataset_builder_modules.preprocess_lyrics import encode_lyrics
from acestep.training.dataset_builder_modules.preprocess_text import encode_text
from acestep.training_v2.model_loader import load_decoder_for_training, load_text_encoder, unload_models

manifest_digest = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
schema = json.dumps({
    'tensor_schema': 'full_song_cover_nofsq_captioned_v1',
    'instruction': TASK_INSTRUCTIONS['cover'],
    'manifest_sha256': manifest_digest,
    'motif_dataset': motif_dataset_config,
}, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
tensor_ready = (
    not RESET_TENSORS
    and TENSOR_SCHEMA_PATH.is_file()
    and TENSOR_SCHEMA_PATH.read_text(encoding='utf-8').strip() == schema
    and next(TENSOR_DIR.glob('*.pt'), None) is not None
)
if RESET_TENSORS and TENSOR_DIR.exists():
    shutil.rmtree(TENSOR_DIR)
TENSOR_DIR.mkdir(parents=True, exist_ok=True)

def build_cover_prompt(caption, duration):
    metas = (
        '- bpm: N/A\n'
        '- timesignature: N/A\n'
        '- keyscale: N/A\n'
        f'- duration: {duration:.1f} seconds\n'
    )
    return SFT_GEN_PROMPT.format(TASK_INSTRUCTIONS['cover'], caption, metas)

if not tensor_ready:
    base_tensor_files = sorted(BASE_TENSOR_DIR.glob('*.pt'))
    assert len(base_tensor_files) == manifest['metadata']['num_samples'], (
        f'재사용할 v7 tensor 수 불일치: {len(base_tensor_files)} / {manifest["metadata"]["num_samples"]}'
    )
    manifest_by_audio = {str(Path(sample['audio_path']).resolve()): sample for sample in manifest['samples']}

    # Pass 1: ACE text encoder로 Cover instruction + caption + 기존 lyrics를 인코딩합니다.
    text_tokenizer, text_encoder = load_text_encoder(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
    for index, source_path in enumerate(base_tensor_files, 1):
        base_item = torch.load(source_path, map_location='cpu', weights_only=True)
        audio_path = str(Path(base_item.get('metadata', {}).get('audio_path', '')).resolve())
        sample = manifest_by_audio.get(audio_path)
        if sample is None:
            raise KeyError(f'manifest sample을 찾지 못했습니다: {source_path.name} / {audio_path}')
        caption = sample['caption']
        caption_hash = hashlib.sha256(caption.encode('utf-8')).hexdigest()
        output_path = TENSOR_DIR / source_path.name
        if output_path.is_file():
            current = torch.load(output_path, map_location='cpu', weights_only=True)
            if (
                current.get('cover_caption_hash') == caption_hash
                and current.get('full_song_motif_schema') == 'cover_nofsq_raw_repeated_motif_mask_ones_v1'
                and 'encoder_hidden_states' in current
                and 'cover_text_hidden_states' not in current
            ):
                continue
        item = base_item
        assert item['target_latents'].shape[-1] == 64
        assert item['context_latents'].shape[-1] == 128
        duration = item['target_latents'].shape[0] / 25.0
        prompt = build_cover_prompt(caption, duration)
        text_hs, text_mask = encode_text(text_encoder, text_tokenizer, prompt, DEVICE, torch.bfloat16)
        lyric_hs, lyric_mask = encode_lyrics(text_encoder, text_tokenizer, sample['lyrics'], DEVICE, torch.bfloat16)
        item['cover_text_hidden_states'] = text_hs.cpu()
        item['cover_text_attention_mask'] = text_mask.cpu()
        item['cover_lyric_hidden_states'] = lyric_hs.cpu()
        item['cover_lyric_attention_mask'] = lyric_mask.cpu()
        item['metadata'] = {**item.get('metadata', {}), 'caption': caption, 'lyrics': sample['lyrics'], 'task_type': 'cover-nofsq'}
        item['cover_caption_hash'] = caption_hash
        item['full_song_motif_schema'] = 'cover_nofsq_raw_repeated_motif_mask_ones_v1'
        temporary = output_path.with_name(output_path.name + '.pass1.tmp')
        torch.save(item, temporary)
        temporary.replace(output_path)
        if index % 25 == 0 or index == len(base_tensor_files):
            print(f'[Pass 1] {index}/{len(base_tensor_files)}')
    unload_models(text_encoder)
    del text_encoder, text_tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    # Pass 2: DiT encoder condition만 갱신하고 임시 text tensor는 제거합니다.
    condition_model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
    output_paths = sorted(TENSOR_DIR.glob('*.pt'))
    for index, output_path in enumerate(output_paths, 1):
        item = torch.load(output_path, map_location='cpu', weights_only=True)
        if 'cover_text_hidden_states' not in item:
            continue
        text_hs = item.pop('cover_text_hidden_states').to(device=DEVICE, dtype=torch.bfloat16)
        text_mask = item.pop('cover_text_attention_mask').to(device=DEVICE)
        lyric_hs = item.pop('cover_lyric_hidden_states').to(device=DEVICE, dtype=torch.bfloat16)
        lyric_mask = item.pop('cover_lyric_attention_mask').to(device=DEVICE)
        encoder_hs, encoder_mask = run_encoder(
            condition_model, text_hs, text_mask, lyric_hs, lyric_mask, DEVICE, torch.bfloat16,
        )
        item['encoder_hidden_states'] = encoder_hs.squeeze(0).cpu()
        item['encoder_attention_mask'] = encoder_mask.squeeze(0).cpu()
        temporary = output_path.with_name(output_path.name + '.pass2.tmp')
        torch.save(item, temporary)
        temporary.replace(output_path)
        if index % 25 == 0 or index == len(output_paths):
            print(f'[Pass 2] {index}/{len(output_paths)}')
    unload_models(condition_model)
    del condition_model
    gc.collect()
    torch.cuda.empty_cache()

    completed = 0
    for output_path in sorted(TENSOR_DIR.glob('*.pt')):
        item = torch.load(output_path, map_location='cpu', weights_only=True)
        assert 'cover_text_hidden_states' not in item
        assert item.get('full_song_motif_schema') == 'cover_nofsq_raw_repeated_motif_mask_ones_v1'
        completed += 1
    assert completed == manifest['metadata']['num_samples'], f'tensor 수 불일치: {completed}'
    TENSOR_SCHEMA_PATH.write_text(schema + '\n', encoding='utf-8')
    tensor_ready = True

tensor_files = sorted(TENSOR_DIR.glob('*.pt'))
assert tensor_ready and tensor_files
example = torch.load(tensor_files[0], map_location='cpu', weights_only=True)
assert example['context_latents'].shape[-1] == 128
assert example['target_latents'].shape[-1] == 64
assert torch.all(example['context_latents'][..., 64:] == 1)
print('full-song tensors:', len(tensor_files))
print('Source+Mask:', tuple(example['context_latents'].shape), 'Target:', tuple(example['target_latents'].shape))

## 3. 단일-stream LoRA 학습

In [ ]:
import gc
import re

LOCAL_TENSOR_DIR = ACESTEP_DIR / 'mohim_full_song_tensors'
LOCAL_RUN_DIR = ACESTEP_DIR / f'mohim_lora_run_{LORA_VERSION}'
if LOCAL_TENSOR_DIR.exists():
    shutil.rmtree(LOCAL_TENSOR_DIR)
shutil.copytree(TENSOR_DIR, LOCAL_TENSOR_DIR)
if RESET_LORA_RUN:
    for path in (LOCAL_RUN_DIR, LORA_RUN_DIR, DRIVE_CHECKPOINT_DIR):
        if path.exists():
            shutil.rmtree(path)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_epoch(path):
    try:
        return int(torch.load(path / 'training_state.pt', map_location='cpu', weights_only=False).get('epoch', -1))
    except Exception:
        return -1

def valid_resume(path):
    required = ('training_state.pt', 'adapter_model.safetensors', 'run_config.json', 'rng_state.pt')
    return path.is_dir() and all((path / name).is_file() for name in required) and checkpoint_epoch(path) >= 0

resume_candidates = [
    DRIVE_CHECKPOINT_DIR / 'resume_latest',
    DRIVE_CHECKPOINT_DIR / 'resume_latest.previous',
    *DRIVE_CHECKPOINT_DIR.glob('epoch_*'),
]
resume_source = max((path for path in resume_candidates if valid_resume(path)), key=checkpoint_epoch, default=None)
resume_local = None
if resume_source is not None:
    resume_local = LOCAL_RUN_DIR / 'resume_checkpoint'
    if resume_local.exists():
        shutil.rmtree(resume_local)
    shutil.copytree(resume_source, resume_local)
    print('resume:', resume_source)
else:
    print('새 full-song 학습을 epoch 1부터 시작합니다.')

### 3.1 Smoke test

full-song tensor 3개로 표준 single-stream LoRA를 1 epoch 학습한 뒤 저장된 adapter를 base model에 다시 로드합니다. 본학습 경로와 완전히 분리됩니다.

In [ ]:
from peft import PeftModel
from acestep.training_v2.model_loader import load_decoder_for_training

SMOKE_SONGS = 3
SMOKE_VERSION = f'{LORA_VERSION}_smoke'
SMOKE_LOCAL_TENSOR_DIR = ACESTEP_DIR / 'mohim_full_song_tensors_smoke'
SMOKE_LOCAL_RUN_DIR = ACESTEP_DIR / f'mohim_lora_run_{SMOKE_VERSION}'
SMOKE_DRIVE_RUN_DIR = LORA_RUN_DIR.parent / SMOKE_VERSION
SMOKE_DRIVE_CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR.parent / SMOKE_VERSION

for path in (SMOKE_LOCAL_TENSOR_DIR, SMOKE_LOCAL_RUN_DIR, SMOKE_DRIVE_RUN_DIR, SMOKE_DRIVE_CHECKPOINT_DIR):
    if path.exists():
        shutil.rmtree(path)
SMOKE_LOCAL_TENSOR_DIR.mkdir(parents=True)

smoke_sources = sorted(LOCAL_TENSOR_DIR.glob('*.pt'))[:SMOKE_SONGS]
assert len(smoke_sources) == SMOKE_SONGS, f'smoke test용 tensor가 부족합니다: {len(smoke_sources)}/{SMOKE_SONGS}'
for source in smoke_sources:
    shutil.copy2(source, SMOKE_LOCAL_TENSOR_DIR / source.name)
print(f'[SMOKE] tensors: {len(smoke_sources)}, output: {SMOKE_LOCAL_RUN_DIR}')

os.environ['MOHIM_CHECKPOINT_BACKUP_DIR'] = str(SMOKE_DRIVE_CHECKPOINT_DIR)
os.environ['PYTHONUNBUFFERED'] = '1'
%cd {ACESTEP_DIR}
!python -u train.py --plain --yes fixed \
  --checkpoint-dir "{CHECKPOINT_DIR}" --model-variant "{MODEL_VARIANT}" \
  --dataset-dir "{SMOKE_LOCAL_TENSOR_DIR}" --output-dir "{SMOKE_LOCAL_RUN_DIR}" \
  --rank 64 --alpha 128 --dropout 0.1 \
  --batch-size 1 --gradient-accumulation 1 \
  --epochs 1 --save-every 1 --lr 0.0003 --val-split 0.2 \
  --shift 1.0 --num-inference-steps 50 \
  --optimizer-type adamw8bit --scheduler-type cosine_restarts \
  --warmup-steps 1 --weight-decay 0.01 --max-grad-norm 1.0 \
  --seed 42 --num-workers 0 --log-every 1 \
  --device "{DEVICE}" --precision bf16
if _exit_code != 0:
    raise RuntimeError(f'smoke training failed (exit={_exit_code})')

smoke_adapter_dir = SMOKE_LOCAL_RUN_DIR / 'final'
assert (smoke_adapter_dir / 'adapter_model.safetensors').is_file(), 'smoke adapter weight가 저장되지 않았습니다.'
assert (smoke_adapter_dir / 'adapter_config.json').is_file(), 'smoke adapter config가 저장되지 않았습니다.'
assert not (smoke_adapter_dir / 'dual_stream_conditioner.pt').exists(), 'single-stream smoke test에 dual-stream conditioner가 생성됐습니다.'

smoke_model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
smoke_model = PeftModel.from_pretrained(smoke_model, smoke_adapter_dir)
print('[OK] Smoke adapter loaded into the base model')
del smoke_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

shutil.copytree(SMOKE_LOCAL_RUN_DIR, SMOKE_DRIVE_RUN_DIR)
SMOKE_TEST_PASSED = True
print(f'[OK] Smoke test passed and copied to {SMOKE_DRIVE_RUN_DIR}')

In [ ]:
assert SMOKE_TEST_PASSED, '먼저 smoke test를 통과해야 합니다.'
os.environ['MOHIM_CHECKPOINT_BACKUP_DIR'] = str(DRIVE_CHECKPOINT_DIR)
os.environ['PYTHONUNBUFFERED'] = '1'
resume_option = f'--resume-from \"{resume_local}\"' if resume_local is not None else ''
%cd {ACESTEP_DIR}
!python -u train.py --plain --yes fixed \
  --checkpoint-dir "{CHECKPOINT_DIR}" --model-variant "{MODEL_VARIANT}" \
  --dataset-dir "{LOCAL_TENSOR_DIR}" --output-dir "{LOCAL_RUN_DIR}" \
  --rank 64 --alpha 128 --dropout 0.1 \
  --batch-size 1 --gradient-accumulation 4 \
  --epochs 100 --save-every 10 --lr 0.0003 --val-split 0.2 \
  --shift 1.0 --num-inference-steps 50 \
  --optimizer-type adamw8bit --scheduler-type cosine_restarts \
  --warmup-steps 100 --weight-decay 0.01 --max-grad-norm 1.0 \
  --seed 42 --num-workers 4 --log-every 10 \
  --device "{DEVICE}" --precision bf16 {resume_option}
if _exit_code != 0:
    raise RuntimeError(f'full-song 학습 실패: {_exit_code}')
FINAL_ADAPTER_DIR = LOCAL_RUN_DIR / 'final'
assert (FINAL_ADAPTER_DIR / 'adapter_model.safetensors').is_file()
shutil.copytree(LOCAL_RUN_DIR, LORA_RUN_DIR, dirs_exist_ok=True)
print('saved:', LORA_RUN_DIR)

## 4. Full-song inference

In [ ]:
import soundfile as sf
from peft import PeftModel
from acestep.constants import SFT_GEN_PROMPT, TASK_INSTRUCTIONS
from acestep.training.dataset_builder_modules.preprocess_encoder import run_encoder
from acestep.training.dataset_builder_modules.preprocess_lyrics import encode_lyrics
from acestep.training.dataset_builder_modules.preprocess_text import encode_text
from acestep.training_v2.dual_stream_preprocess import _encode_audio, encode_motif_condition_latents
from acestep.training_v2.model_loader import load_decoder_for_training, load_text_encoder, load_vae, unload_models

CUSTOM_MOTIF_PATH = Path('/content/motif.wav')
CUSTOM_MOTIF_STEM = 'piano'
CUSTOM_MOTIF_START_SECONDS = 0.0
CUSTOM_CAPTION = '''
Paste a detailed ACE-Step-style full-song caption here.
'''
CUSTOM_LYRICS = '''
Paste the complete lyrics here.
'''
CUSTOM_OUTPUT_NAME = 'custom_full_song'
INFERENCE_DURATION_SECONDS = 180.0  # 원하는 전체 곡 길이로 수정, 최대 300초 권장
INFERENCE_STEPS = 50
INFERENCE_SEED = 42
CFG_SCALE = 7.0
EXPERIMENTS = ('motif',)  # 비교가 필요하면 ('motif', 'no_motif')
CHECKPOINT_OVERRIDE = None

assert CUSTOM_MOTIF_PATH.is_file()
assert CUSTOM_CAPTION.strip() and 'Paste a detailed' not in CUSTOM_CAPTION
assert CUSTOM_LYRICS.strip() and 'Paste the complete lyrics' not in CUSTOM_LYRICS

def inference_recurring_motif_sentence(motif_stem):
    stem = str(motif_stem).strip().lower().replace('_', ' ')
    return (
        f'This full song is built around the provided {stem} motif. '
        f'The clear, distinctive, and recognizable {stem} motif serves as '
        "the song's central musical hook and recurring signature, returning "
        'prominently and frequently throughout the entire song—not only in the '
        'intro, but across verses, choruses, transitions, and instrumental passages—'
        'while remaining clearly audible within the full arrangement and creating '
        'a strong sense of continuity.'
    )

def build_inference_cover_prompt(caption, duration):
    metas = (
        '- bpm: N/A\n- timesignature: N/A\n- keyscale: N/A\n'
        f'- duration: {duration:.1f} seconds\n'
    )
    return SFT_GEN_PROMPT.format(TASK_INSTRUCTIONS['cover'], caption, metas)
checkpoint_pattern = re.compile(r'^epoch_(\d+)_loss_([0-9.]+)$')
checkpoint_candidates = []
for path in DRIVE_CHECKPOINT_DIR.glob('epoch_*_loss_*'):
    match = checkpoint_pattern.match(path.name)
    if match and (path / 'adapter_model.safetensors').is_file():
        checkpoint_candidates.append((float(match.group(2)), int(match.group(1)), path))
for path in (DRIVE_CHECKPOINT_DIR / 'resume_latest', DRIVE_CHECKPOINT_DIR / 'resume_latest.previous'):
    config_path = path / 'run_config.json'
    if (path / 'adapter_model.safetensors').is_file() and config_path.is_file():
        loss = json.loads(config_path.read_text(encoding='utf-8')).get('val_loss')
        epoch = checkpoint_epoch(path)
        if loss is not None and epoch >= 0:
            checkpoint_candidates.append((float(loss), epoch, path))
if CHECKPOINT_OVERRIDE is not None:
    BEST_ADAPTER_DIR = Path(CHECKPOINT_OVERRIDE)
    BEST_LOSS, BEST_EPOCH = None, checkpoint_epoch(BEST_ADAPTER_DIR)
else:
    assert checkpoint_candidates, f'checkpoint가 없습니다: {DRIVE_CHECKPOINT_DIR}'
    BEST_LOSS, BEST_EPOCH, BEST_ADAPTER_DIR = min(checkpoint_candidates, key=lambda row: row[0])
print('checkpoint:', BEST_ADAPTER_DIR, 'epoch:', BEST_EPOCH, 'loss:', BEST_LOSS)

In [ ]:
dtype = torch.bfloat16
generation_duration = INFERENCE_DURATION_SECONDS
target_samples = round(generation_duration * 48000)
motif_duration = sf.info(str(CUSTOM_MOTIF_PATH)).duration
vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
motif_source = encode_motif_condition_latents(
    str(CUSTOM_MOTIF_PATH), vae, dtype, target_samples=target_samples,
    motif_start_sec=CUSTOM_MOTIF_START_SECONDS,
    motif_end_sec=CUSTOM_MOTIF_START_SECONDS + motif_duration,
)
silence_source = _encode_audio(torch.zeros(2, target_samples), vae, dtype) if 'no_motif' in EXPERIMENTS else None
unload_models(vae)
del vae
assert silence_source is None or motif_source.shape == silence_source.shape

caption = f'{CUSTOM_CAPTION.strip()} {inference_recurring_motif_sentence(CUSTOM_MOTIF_STEM)}'
text_tokenizer, text_encoder = load_text_encoder(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
prompt = build_inference_cover_prompt(caption, generation_duration)
text_hs, text_mask = encode_text(text_encoder, text_tokenizer, prompt, DEVICE, dtype)
lyric_hs, lyric_mask = encode_lyrics(text_encoder, text_tokenizer, CUSTOM_LYRICS.strip(), DEVICE, dtype)
unload_models(text_encoder)
del text_encoder, text_tokenizer

model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
encoder_hs, encoder_mask = run_encoder(model, text_hs, text_mask, lyric_hs, lyric_mask, DEVICE, dtype)
del text_hs, text_mask, lyric_hs, lyric_mask
model.decoder = PeftModel.from_pretrained(model.decoder, str(BEST_ADAPTER_DIR), is_trainable=False).to(device=DEVICE, dtype=dtype).eval()

@torch.no_grad()
def sample_full_song(source_latents):
    generator = torch.Generator(device=DEVICE).manual_seed(INFERENCE_SEED)
    latents = torch.randn((1, *source_latents.shape), device=DEVICE, dtype=dtype, generator=generator)
    source = source_latents.unsqueeze(0).to(DEVICE, dtype=dtype)
    source_mask = torch.ones_like(source)
    context = torch.cat([source, source_mask], dim=-1)
    attention_mask = torch.ones(1, source.shape[1], device=DEVICE, dtype=dtype)
    condition = encoder_hs.to(DEVICE, dtype=dtype)
    condition_mask = encoder_mask.to(DEVICE, dtype=dtype)
    unconditional = model.null_condition_emb.to(DEVICE, dtype=dtype).expand_as(condition)
    times = torch.linspace(1.0, 0.0, INFERENCE_STEPS + 1, device=DEVICE, dtype=dtype)
    for index in range(INFERENCE_STEPS):
        timestep = times[index].expand(1)
        combined = model.decoder(
            hidden_states=torch.cat([latents, latents]),
            timestep=torch.cat([timestep, timestep]), timestep_r=torch.cat([timestep, timestep]),
            attention_mask=torch.cat([attention_mask, attention_mask]),
            encoder_hidden_states=torch.cat([condition, unconditional]),
            encoder_attention_mask=torch.cat([condition_mask, condition_mask]),
            context_latents=torch.cat([context, context]),
        )[0]
        conditional_velocity, unconditional_velocity = combined.chunk(2)
        velocity = unconditional_velocity + CFG_SCALE * (conditional_velocity - unconditional_velocity)
        latents = latents + (times[index + 1] - times[index]) * velocity
    return latents.cpu()

generated_latents = {}
for experiment in EXPERIMENTS:
    source = motif_source if experiment == 'motif' else silence_source
    generated_latents[experiment] = sample_full_song(source)
    print(experiment, 'complete')
del model, motif_source, silence_source, encoder_hs, encoder_mask
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import math
import torch.nn.functional as F
from IPython.display import Audio, display

def decode_latents_tiled(vae, latents_btc, chunk_frames=256, overlap=64):
    latents = latents_btc.transpose(1, 2)
    stride = chunk_frames - 2 * overlap
    decoded, factor = [], None
    for index in range(math.ceil(latents.shape[-1] / stride)):
        core_start = index * stride
        core_end = min(core_start + stride, latents.shape[-1])
        window_start = max(0, core_start - overlap)
        window_end = min(latents.shape[-1], core_end + overlap)
        chunk = latents[:, :, window_start:window_end].to(DEVICE, dtype=vae.dtype)
        with torch.inference_mode():
            audio = vae.decode(chunk).sample
        factor = factor or audio.shape[-1] / chunk.shape[-1]
        trim_start = round((core_start - window_start) * factor)
        trim_end = round((window_end - core_end) * factor)
        end = audio.shape[-1] - trim_end if trim_end else audio.shape[-1]
        decoded.append(audio[:, :, trim_start:end].float().cpu())
    return torch.cat(decoded, dim=-1)

vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
outputs = {}
for experiment, latents in generated_latents.items():
    audio = decode_latents_tiled(vae, latents)
    if audio.shape[-1] < target_samples:
        audio = F.pad(audio, (0, target_samples - audio.shape[-1]))
    audio = audio[:, :, :target_samples]
    audio = audio / audio.abs().amax().clamp_min(1.0)
    output_dir = INFERENCE_OUTPUT_DIR / BEST_ADAPTER_DIR.name / CUSTOM_OUTPUT_NAME / experiment
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'generated_full_song.wav'
    sf.write(path, audio.squeeze(0).transpose(0, 1).numpy(), 48000)
    outputs[experiment] = path
    print(experiment, path)
    display(Audio(filename=str(path)))
unload_models(vae)
del vae, generated_latents
gc.collect()
torch.cuda.empty_cache()

## 5. Generated full song ↔ motif 유사도

In [ ]:
import numpy as np
import pandas as pd
import librosa
from sklearn.metrics.pairwise import cosine_similarity
from acestep.training.dataset_builder_modules.preprocess_audio import load_audio_stereo
from acestep.training_v2.dual_stream_preprocess import build_repeated_motif_canvas
from mohim.motif import _max_shifted_cosine_similarity, _resize_time

def features(audio):
    hop, n_fft = 512, 2048
    preroll = n_fft // hop
    padded = np.pad(audio, (preroll * hop, 0))
    padded_onset = librosa.onset.onset_strength(y=padded, sr=48000, hop_length=hop, n_fft=n_fft)
    rms = librosa.feature.rms(y=audio, hop_length=hop)[0]
    onset = _resize_time(padded_onset[preroll:preroll + len(rms)][None], 256).ravel()
    chroma = _resize_time(librosa.feature.chroma_cens(y=audio, sr=48000, hop_length=hop), 64)
    centered_chroma = chroma - chroma.mean(axis=0, keepdims=True)
    return onset, chroma.ravel(), centered_chroma.ravel()

motif_audio, _ = load_audio_stereo(str(CUSTOM_MOTIF_PATH), 48000, generation_duration)
reference = build_repeated_motif_canvas(
    motif_audio, target_samples, motif_start_sec=CUSTOM_MOTIF_START_SECONDS,
    motif_end_sec=CUSTOM_MOTIF_START_SECONDS + motif_duration, sample_rate=48000,
).mean(0).numpy()
reference_onset, reference_chroma, reference_centered_chroma = features(reference)
rows = []
for experiment, path in outputs.items():
    generated, _ = librosa.load(str(path), sr=48000, mono=True)
    onset, chroma, centered_chroma = features(generated)
    raw_onset = _max_shifted_cosine_similarity(reference_onset, onset, 3, cosine_similarity, mean_center=False)
    centered_onset = _max_shifted_cosine_similarity(reference_onset, onset, 3, cosine_similarity, mean_center=True)
    raw_chroma = float(cosine_similarity(reference_chroma[None], chroma[None])[0, 0])
    centered_chroma_score = float(cosine_similarity(reference_centered_chroma[None], centered_chroma[None])[0, 0])
    rows.append({
        'experiment': experiment, 'onset_similarity': raw_onset, 'chroma_similarity': raw_chroma,
        'mean_centered_onset_similarity': centered_onset,
        'mean_centered_chroma_similarity': centered_chroma_score,
        'similarity': 0.7 * centered_onset + 0.3 * centered_chroma_score,
    })
table = pd.DataFrame(rows)
display(table.style.format({column: '{:.4f}' for column in table.columns if column != 'experiment'}))
score_dir = INFERENCE_OUTPUT_DIR / BEST_ADAPTER_DIR.name / CUSTOM_OUTPUT_NAME
table.to_csv(score_dir / 'full_song_motif_similarity.csv', index=False)
(score_dir / 'full_song_motif_similarity.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')